# Experiment - FEVER
Main sweep: 3 models × 3 prompt types × {r=0, r=1} on FEVER (N=50).
Results saved to `results/fever_results.csv`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import yaml
import pandas as pd
from pathlib import Path

with open("../configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

MODELS       = cfg["models"]["available"]
PROMPT_TYPES = ["standard", "chain_of_thought", "vigilant"]
N            = cfg["evaluation"]["n_examples"]
SEED         = cfg["seed"]
RESULTS_DIR  = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Models: {MODELS}")
print(f"Prompt types: {PROMPT_TYPES}")
print(f"N: {N}  seed: {SEED}  k: {cfg['retrieval']['k']}")

Models: ['Qwen/Qwen3.5-2B', 'google/gemma-4-E2B-it']
Prompt types: ['standard', 'chain_of_thought', 'vigilant']
N: 75  seed: 42  k: 10


In [2]:
from src.data.fever_loader import load_fever

POOL_SIZE = 200  # fixed pool so cache stays valid when N changes
all_clean = load_fever("../" + cfg["dataset"]["fever_dev"], max_examples=POOL_SIZE)
examples = all_clean[:N]  # same Python objects — identity-based exclusion works
print(f"Pool: {len(all_clean)}  Evaluated: {len(examples)}")
print("Label distribution:", {l: sum(1 for e in examples if e["label"] == l)
                              for l in {e["label"] for e in examples}})

Pool: 200  Evaluated: 75
Label distribution: {'NOT ENOUGH INFO': 26, 'SUPPORTS': 24, 'REFUTES': 25}


In [3]:
from src.retrieval.embedder import Embedder
from src.retrieval.retriever import Retriever

emb_cache = os.path.join("../", cfg["cache"]["dir"], cfg["cache"]["embeddings_subdir"])
embedder = Embedder(model_name=cfg["retrieval"]["embedding_model"], cache_dir=emb_cache)
retriever = Retriever(embedder=embedder, k=cfg["retrieval"]["k"])
print(f"Embedder: {cfg['retrieval']['embedding_model']}  k={cfg['retrieval']['k']}")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Embedder: IEITYuan/Yuan-embedding-2.0-en  k=10


In [4]:
from src.data.fever_loader import load_fever
from src.evaluation.scorer import run as run_scorer
from src.generation.llm_client import HuggingFaceClient

all_poisoned = load_fever("../" + cfg["dataset"]["fever_dev_poisoned"])
examples_poisoned = all_poisoned[:N]  # same objects — identity exclusion works
print(f"Clean examples   : {len(examples)}")
print(f"Poisoned examples: {len(examples_poisoned)}")

records = []

for model in MODELS:
    print(f"\n{'='*60}\nModel: {model}\n{'='*60}")
    llm_cache = os.path.join("../", cfg["cache"]["dir"], cfg["cache"]["llm_subdir"])
    llm = HuggingFaceClient(
        model=model,
        temperature=cfg["models"]["temperature"],
        cache_dir=llm_cache,
    )
    with embedder, llm:
        for prompt_type in PROMPT_TYPES:
            print(f"  prompt={prompt_type}  r=0", end="")
            m0 = run_scorer(
                examples=examples, retriever=retriever, llm=llm,
                prompt_type=prompt_type, seed=SEED,
                full_dataset=all_clean,
            )
            records.append({"model": model, "prompt_type": prompt_type,
                            "poison_rate": 0, **m0})
            print(f" acc={m0['accuracy']:.3f}  r=1", end="")
            m1 = run_scorer(
                examples=examples_poisoned, retriever=retriever, llm=llm,
                prompt_type=prompt_type, seed=SEED,
                full_dataset=all_poisoned,
            )
            records.append({"model": model, "prompt_type": prompt_type,
                            "poison_rate": 1, **m1})
            print(f" acc={m1['accuracy']:.3f}")

print("\nSweep complete.")

Clean examples   : 75
Poisoned examples: 75

Model: Qwen/Qwen3.5-2B
  prompt=standard  r=0 acc=0.640  r=1 acc=0.587
  prompt=chain_of_thought  r=0 acc=0.560  r=1 acc=0.413
  prompt=vigilant  r=0 acc=0.453  r=1 acc=0.413

Model: google/gemma-4-E2B-it
  prompt=standard  r=0 acc=0.573  r=1 acc=0.467
  prompt=chain_of_thought  r=0 acc=0.480  r=1 acc=0.440
  prompt=vigilant  r=0 acc=0.520  r=1 acc=0.467

Sweep complete.


In [5]:
df = pd.DataFrame(records)
out_path = RESULTS_DIR / "fever_results.csv"
df.to_csv(out_path, index=False)
print(f"Saved {len(df)} rows to {out_path}")
df

Saved 12 rows to ../results/fever_results.csv


,model,prompt_type,poison_rate,accuracy,macro_f1,hallucination_rate,contradiction_detection_rate
0,Qwen/Qwen3.5-2B,standard,0,0.640000,0.625749,0.576923,NaN
1,Qwen/Qwen3.5-2B,standard,1,0.586667,0.584049,0.423077,NaN
2,Qwen/Qwen3.5-2B,chain_of_thought,0,0.560000,0.532846,0.115385,NaN
3,Qwen/Qwen3.5-2B,chain_of_thought,1,0.413333,0.363564,0.230769,NaN
4,Qwen/Qwen3.5-2B,vigilant,0,0.453333,0.412960,0.884615,0.000000
5,Qwen/Qwen3.5-2B,vigilant,1,0.413333,0.400386,0.769231,0.000000
6,google/gemma-4-E2B-it,standard,0,0.573333,0.538501,0.807692,NaN
7,google/gemma-4-E2B-it,standard,1,0.466667,0.432003,0.846154,NaN
8,google/gemma-4-E2B-it,chain_of_thought,0,0.480000,0.475439,0.307692,NaN
9,google/gemma-4-E2B-it,chain_of_thought,1,0.440000,0.373074,0.076923,NaN
